# PEFT 下一步 — Adapter 多模态复活 + 路线图

配套 lecture: [../lectures/10-peft-next-step.md](../lectures/10-peft-next-step.md)

本 notebook 演示:
1. LLaMA-Adapter zero-init attention pseudo-code
2. Q-Former 架构示意
3. 后续专题预告

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. LLaMA-Adapter — Zero-init Attention

核心 idea: 在 attention 上加 prefix tokens，但用一个 learnable gate 控制其影响。
初始 gate=0 → 模型行为 = 原始 LLaMA，逐渐学习 gate 让 prefix 生效。

In [2]:
class LLaMAAdapterAttention(nn.Module):
    """LLaMA-Adapter 的 zero-init attention 简化版。"""
    def __init__(self, d, n_heads, prefix_len=10):
        super().__init__()
        self.d = d
        self.n_heads = n_heads
        # 学习的 prefix tokens (K, V)
        self.prefix_k = nn.Parameter(torch.randn(prefix_len, d) * 0.02)
        self.prefix_v = nn.Parameter(torch.randn(prefix_len, d) * 0.02)
        # Zero-init gate (核心!)
        self.gate = nn.Parameter(torch.zeros(n_heads))

    def forward(self, q, k, v):
        """q, k, v: (batch, seq, d)"""
        # 原 attention
        attn_orig = (q @ k.transpose(-1, -2)) / (self.d ** 0.5)
        attn_orig = F.softmax(attn_orig, dim=-1)
        out_orig = attn_orig @ v

        # Prefix attention
        attn_pref = (q @ self.prefix_k.T) / (self.d ** 0.5)
        attn_pref = F.softmax(attn_pref, dim=-1)
        out_pref = attn_pref @ self.prefix_v

        # Zero-init gate: 初始 gate=0 -> 只有 out_orig 生效
        gate_avg = self.gate.mean()  # 简化为标量
        return out_orig + gate_avg * out_pref

# 验证初始 forward = 原 attention
torch.manual_seed(0)
model = LLaMAAdapterAttention(d=16, n_heads=4, prefix_len=10)
q = torch.randn(2, 5, 16); k = torch.randn(2, 5, 16); v = torch.randn(2, 5, 16)

# Zero-init: gate = 0
out = model(q, k, v)

# 手算原 attention
attn_orig = F.softmax((q @ k.transpose(-1, -2)) / 4, dim=-1)
out_orig_only = attn_orig @ v

diff = (out - out_orig_only).abs().max().item()
print(f'初始 LLaMA-Adapter forward vs 原 attention: {diff:.4e}')
print(f'gate 初值: {model.gate.tolist()}')
print(f'\n→ Zero-init gate 保证初始模型行为不变')
print(f'→ 训练时 gate 逐渐学习，prefix 逐渐生效')

初始 LLaMA-Adapter forward vs 原 attention: 0.0000e+00
gate 初值: [0.0, 0.0, 0.0, 0.0]

→ Zero-init gate 保证初始模型行为不变
→ 训练时 gate 逐渐学习，prefix 逐渐生效


## 2. Q-Former (BLIP-2) — 架构示意

In [3]:
class QFormer(nn.Module):
    """BLIP-2 Q-Former 极简版。
    
    输入: image_features (b, n_img, d_img)
    输出: visual_tokens (b, n_query, d_llm)
    """
    def __init__(self, n_queries=32, d_img=1024, d_llm=4096):
        super().__init__()
        # 32 个 learnable query tokens
        self.queries = nn.Parameter(torch.randn(n_queries, d_img) * 0.02)
        # Cross-attention: queries attend to image features
        self.cross_attn = nn.MultiheadAttention(d_img, num_heads=8, batch_first=True)
        # 投影到 LLM 维度
        self.proj = nn.Linear(d_img, d_llm)

    def forward(self, image_features):
        """image_features: (b, n_img, d_img)"""
        b = image_features.size(0)
        # 复制 queries 到 batch
        q = self.queries.unsqueeze(0).expand(b, -1, -1)  # (b, 32, d_img)
        # Cross-attention: q queries the image
        attn_out, _ = self.cross_attn(q, image_features, image_features)
        # 投影到 LLM dim
        return self.proj(attn_out)  # (b, 32, d_llm)

qformer = QFormer(n_queries=32, d_img=1024, d_llm=4096)
image_features = torch.randn(2, 257, 1024)  # ViT 输出 (1 CLS + 256 patches)
visual_tokens = qformer(image_features)

print(f'输入 (image features):  {image_features.shape}')
print(f'输出 (visual tokens):   {visual_tokens.shape}')
print(f'\nQ-Former 把 257 image patches 压缩到 32 visual tokens')
print(f'再投影到 LLM dim (4096) → 喂给 frozen LLM')

n_params = sum(p.numel() for p in qformer.parameters())
print(f'\nQ-Former 参数: {n_params:,}')
print(f'→ 这是 PEFT 在多模态时代的标准实现')

输入 (image features):  torch.Size([2, 257, 1024])
输出 (visual tokens):   torch.Size([2, 32, 4096])

Q-Former 把 257 image patches 压缩到 32 visual tokens
再投影到 LLM dim (4096) → 喂给 frozen LLM

Q-Former 参数: 8,429,568
→ 这是 PEFT 在多模态时代的标准实现


## 3. LLaVA Projector — 最简 adapter

In [4]:
class LLaVAProjector(nn.Module):
    """LLaVA-1.5 的 projector：单层 MLP。"""
    def __init__(self, d_img=1024, d_llm=4096):
        super().__init__()
        # LLaVA-1.5 使用 2-layer MLP
        self.fc1 = nn.Linear(d_img, d_llm)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(d_llm, d_llm)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

projector = LLaVAProjector(d_img=1024, d_llm=4096)
x = torch.randn(2, 576, 1024)
out = projector(x)
n_params = sum(p.numel() for p in projector.parameters())
print(f'输入 (image patches): {x.shape}')
print(f'输出 (LLM tokens):    {out.shape}')
print(f'参数量: {n_params:,}')
print(f'\n→ 这就是 "adapter" — 最简单的视觉-语言桥')
print(f'→ 视觉 ViT 完全冻结，LLM 完全冻结，只训这个 projector')

输入 (image patches): torch.Size([2, 576, 1024])
输出 (LLM tokens):    torch.Size([2, 576, 4096])
参数量: 20,979,712

→ 这就是 "adapter" — 最简单的视觉-语言桥
→ 视觉 ViT 完全冻结，LLM 完全冻结，只训这个 projector


## 4. 后续专题预告

In [5]:
next_topics = [
    ('对齐 (Alignment)', '⭐⭐⭐', 'RLHF / DPO / SimPO - 让模型符合人类偏好'),
    ('长上下文 (Long Context)', '⭐⭐', 'LongLoRA / PI / YaRN - 128K+ context'),
    ('MoE', '⭐⭐', 'Mixtral / DeepSeek-MoE - 稀疏激活专家'),
    ('多模态', '⭐⭐', 'LLaVA / Q-Former 深入 - 视觉语言对齐'),
    ('推理优化', '⭐', 'vLLM / FlashAttention - 部署级优化'),
    ('推理 (Reasoning)', '⭐⭐', 'CoT / ToT / RAG - 复杂推理能力'),
]

print(f'{"专题":<22}{"优先级":<10}{"说明":<50}')
print('-' * 85)
for name, pri, desc in next_topics:
    print(f'{name:<22}{pri:<10}{desc:<50}')

print(f'\n推荐: 接续书本主线 (Ch3 对齐) → RLHF / DPO / SimPO')

专题                    优先级       说明                                                
-------------------------------------------------------------------------------------
对齐 (Alignment)        ⭐⭐⭐       RLHF / DPO / SimPO - 让模型符合人类偏好                    
长上下文 (Long Context)   ⭐⭐        LongLoRA / PI / YaRN - 128K+ context              
MoE                   ⭐⭐        Mixtral / DeepSeek-MoE - 稀疏激活专家                   
多模态                   ⭐⭐        LLaVA / Q-Former 深入 - 视觉语言对齐                      
推理优化                  ⭐         vLLM / FlashAttention - 部署级优化                     
推理 (Reasoning)        ⭐⭐        CoT / ToT / RAG - 复杂推理能力                          

推荐: 接续书本主线 (Ch3 对齐) → RLHF / DPO / SimPO


## 5. 终章思考

经过三个 PEFT 专题（Prompt + LoRA + Adapter，共 28 方法、~29 小时），你应该:

1. **概念**: 理解 PEFT 三大主线的本质统一
2. **数学**: 推导任何方法的参数量和梯度
3. **代码**: 用 30 行手写出 LoRA / Pfeiffer / (IA)³
4. **工程**: 给定场景选出最优 PEFT 方法

**结束语**: 

PEFT 是大模型时代的 "foundation skill"。掌握它，意味着你可以:
- 在单 GPU 上微调 65B+ 模型
- 构建 multi-task / multi-domain PEFT 服务
- 把任何新论文的 PEFT 方法在 1 小时内复现

**下一站**: 对齐专题 (RLHF / DPO / SimPO)

祝学习愉快！🚀